In [ ]:
# ============================================================================
# CELL 1: Chunked collection + incremental parquet write
# ============================================================================

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import json
import time
from datetime import datetime, timezone
from pathlib import Path

from clinical_trial_benchmarking.taxonomy import THERAPEUTIC_AREA_EXTENDED

In [ ]:
# ---------------------------------------------------------------------------
# 1. Build the query from taxonomy
# ---------------------------------------------------------------------------
_all_terms = set()
for ta_name, keywords in THERAPEUTIC_AREA_EXTENDED.items():
    for kw in keywords:
        term = ' '.join(kw.lower().strip().split())
        if term:
            _all_terms.add(term)

_terms_sorted = sorted(_all_terms)
def _q(t): return f'"{t}"' if ' ' in t else t
query_cond = ' OR '.join(_q(t) for t in _terms_sorted)
print(f"Built query.cond from {len(THERAPEUTIC_AREA_EXTENDED)} TAs, "
      f"{len(_terms_sorted)} unique terms")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Request setup
# ---------------------------------------------------------------------------
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
params = {"query.cond": query_cond, "pageSize": 1000, "format": "json"}

MAX_PAGES = 1000
REQUEST_TIMEOUT = 60
MAX_RETRIES = 5
RETRY_BACKOFF = 2.0

In [ ]:
# ---------------------------------------------------------------------------
# 3. Output paths
# ---------------------------------------------------------------------------
OUT_PATH = Path('clinical_trials_major_diseases.parquet')
META_PATH = OUT_PATH.with_suffix('.metadata.json')
PARTIAL_FLAG = OUT_PATH.with_suffix('.parquet.partial')

# Remove any stale partial from a prior failed run.
if PARTIAL_FLAG.exists():
    PARTIAL_FLAG.unlink()

In [ ]:
# ---------------------------------------------------------------------------
# 4. Chunked collection + incremental writer
# ---------------------------------------------------------------------------
# Strategy:
#   - Accumulate page results in a buffer of N pages.
#   - When the buffer flushes, run json_normalize on it, coerce to a stable
#     schema, and append to a parquet writer.
#   - The buffer is sized so that ONE normalize call fits comfortably in RAM.
#
# A 5-page buffer (5,000 records) is conservative. If your machine has more
# headroom, 20 pages (20,000 records) is still safe and much faster. 
#
# Two-pass schema: the first flush determines the schema; subsequent flushes
# are reindexed to match. This avoids pyarrow complaining about schema drift
# when a later page contains a field that an earlier page omitted (common
# with optional CT.gov modules).

BUFFER_PAGES = 20
session = requests.Session()
writer = None
parquet_schema = None
schema_columns = None
total_written = 0
page_num = 0
page_token = None

def _flush_buffer(buffer):
    """Normalize a list of records and append to the parquet writer."""
    global writer, parquet_schema, schema_columns, total_written

    # json_normalize on the buffer only — bounded memory.
    chunk = pd.json_normalize(buffer)

    if schema_columns is None:
        # First flush: establish schema.
        schema_columns = list(chunk.columns)
        # Cast everything to a pandas-safe type. Object dtype is the killer;
        # convert bools and nullable ints to something parquet handles well.
        for c in chunk.columns:
            if chunk[c].dtype == object:
                # Leave genuinely string-like columns as object; pyarrow
                # handles them as strings.
                pass
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        parquet_schema = table.schema
        writer = pq.ParquetWriter(PARTIAL_FLAG, parquet_schema,
                                  compression='snappy')
    else:
        # Later flushes: align columns to the established schema.
        missing = [c for c in schema_columns if c not in chunk.columns]
        for c in missing:
            chunk[c] = None
        chunk = chunk[schema_columns]

    # Force schema alignment. If a column drifted type (e.g., int in one
    # page, float in another), this is where it gets caught.
    table = pa.Table.from_pandas(chunk, preserve_index=False,
                                 schema=parquet_schema)
    writer.write_table(table)
    total_written += len(chunk)

    # Free the chunk before returning.
    del chunk, table

buffer = []
while True:
    page_num += 1
    if page_num > MAX_PAGES:
        raise RuntimeError(
            f"Exceeded MAX_PAGES={MAX_PAGES}. "
            f"Collected {total_written:,} records so far."
        )

    if page_token:
        params["pageToken"] = page_token

    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(BASE_URL, params=params,
                                   timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            last_err = None
            break
        except requests.exceptions.RequestException as e:
            last_err = e
            wait = RETRY_BACKOFF ** attempt
            print(f"  Page {page_num}, attempt {attempt+1}/{MAX_RETRIES} "
                  f"failed: {e}. Retrying in {wait:.1f}s ...")
            time.sleep(wait)

    if last_err is not None:
        raise RuntimeError(
            f"Page {page_num}: all retries failed. "
            f"Last error: {last_err}. "
            f"Wrote {total_written:,} records to {PARTIAL_FLAG}."
        ) from last_err

    studies = data.get("studies", [])
    buffer.extend(studies)

    if len(buffer) >= BUFFER_PAGES * 1000 or not data.get("nextPageToken"):
        _flush_buffer(buffer)
        buffer = []
        print(f"  Page {page_num:>4}: flushed. "
              f"Total written: {total_written:,}")

    page_token = data.get("nextPageToken")
    if not page_token:
        break

if writer is not None:
    writer.close()

In [ ]:
# ---------------------------------------------------------------------------
# 5. Atomic promote + metadata
# ---------------------------------------------------------------------------
PARTIAL_FLAG.replace(OUT_PATH)

metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "api_endpoint": BASE_URL,
    "query.cond": query_cond,
    "taxonomy_source":
        "clinical_trial_forecasting.taxonomy.THERAPEUTIC_AREA_EXTENDED",
    "n_taxonomy_terms": len(_terms_sorted),
    "pages_collected": page_num,
    "total_records": total_written,
    "complete": True,
    "output_path": str(OUT_PATH),
    "output_size_bytes": OUT_PATH.stat().st_size,
}
with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nWrote {total_written:,} rows to {OUT_PATH}")
print(f"Metadata → {META_PATH}")